# Cricket-Shot CNN-LSTM — Training Notebook
**Prerequisites:** Attach the `cricket-shot` dataset as an input.

Pipeline:
1. Preprocess the data if not available on disk
2. Load ALL frames into RAM once → zero disk I/O during training
3. CNN Spatial Encoder → Bi-LSTM → Temporal Attention → Classifier
4. tqdm progress for epoch + batch loops
5. Minimal JSON logging (loss / acc / lr)
6. Activations & internals available on-demand during inference
---

## Install & GPU Check

In [ ]:
!pip install torch torchvision torchaudio
!pip install pandas matplotlib seaborn numpy scipy tqdm opencv-python

In [5]:
# torchinfo gives a nice model summary — everything else is pre-installed on Kaggle
import subprocess
# subprocess.run(['pip', 'install', 'torchinfo', '-q'])

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch  : 2.11.0+cu130
CUDA     : False


## Config

In [ ]:
import torch, os
import json
import numpy as np

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DATA_DIR = './cricket-shot-dataset'
LOG_DIR          = './logs'
CKPT_DIR         = './checkpoints'
os.makedirs(LOG_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

IDX_TO_CLASS = {
    '0': 'cover',
    '1': 'defense',
    '2': 'flick',
    '3': 'hook',
    '4': 'late_cut',
    '5': 'lofted',
    '6': 'pull',
    '7': 'square_cut',
    '8': 'straight',
    '9': 'sweep',
}    

# ── Classes ───────────────────────────────────────────────────────────────────
SPLITS = ['train', 'val', 'test']
CLASS_NAMES = list(IDX_TO_CLASS.values())
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c  in enumerate(CLASS_NAMES)}

# ── Preprocessing constants (must match preprocess.ipynb) ─────────────────────
NUM_FRAMES = 16
FRAME_H    = 224
FRAME_W    = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
MEAN = np.array(IMAGENET_MEAN, dtype=np.float32).reshape(1, 1, 1, 3)  # (1,1,1,3)
STD  = np.array(IMAGENET_STD,  dtype=np.float32).reshape(1, 1, 1, 3)

# ── Model ─────────────────────────────────────────────────────────────────────
CNN_CHANNELS = [32, 64, 128, 256]
CNN_PROJ_DIM = 256
LSTM_HIDDEN  = 256
LSTM_LAYERS  = 1
CLF_HIDDEN   = 128
CLF_DROPOUT  = 0.5

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE       = 16
NUM_EPOCHS       = 100
LEARNING_RATE    = 1e-4
CHECKPOINT_EVERY = 5
SEED             = 42


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device  : {DEVICE}")
print(f"Classes : {NUM_CLASSES}, {CLASS_NAMES}")
print(f"Batch   : {BATCH_SIZE}  |  Epochs: {NUM_EPOCHS}")


Device  : cpu
Classes : 10, ['cover', 'defense', 'flick', 'hook', 'late_cut', 'lofted', 'pull', 'square_cut', 'straight', 'sweep']
Batch   : 16  |  Epochs: 100


# Preprocessing

In [9]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
from pathlib import Path
import cv2
import numpy as np
import json
import time
from collections import defaultdict

# Define dynamic path based on current parameters
PREPROCESSED_DIR = f"./preprocessed_{NUM_FRAMES}_{FRAME_H}_{FRAME_W}"

def extract_frames(video_path: str, n: int, h: int, w: int):
    """
    Uniformly sample n frames from a video, resize to (h, w).
    Returns uint8 array of shape (n, h, w, 3), or None on failure.
    """
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return None

    ## Linear sampling
    indices = np.linspace(0, max(total - 1, 0), n, dtype=int)
    
    frames, last = [], np.zeros((h, w, 3), dtype=np.uint8)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            last = cv2.resize(
                cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (w, h),
                interpolation=cv2.INTER_LINEAR,
            )
        frames.append(last.copy())

    cap.release()
    return np.stack(frames, axis=0)   # (n, h, w, 3)  uint8

def preprocess_dataset(force=False):
    if os.path.exists(PREPROCESSED_DIR) and not force:
        print(f"Found existing preprocessed data at {PREPROCESSED_DIR}. Skipping.")
        return

    print(f"Preprocessing videos to {PREPROCESSED_DIR}...")
    os.makedirs(PREPROCESSED_DIR, exist_ok=True)

    tasks = []
    for split in ['train', 'val', 'test']:
        base = Path(RAW_DATA_DIR) / split
        out_split = Path(PREPROCESSED_DIR) / split
        
        for cls in CLASS_NAMES:
            cls_dir = base / cls
            if not cls_dir.exists(): continue
            (out_split / cls).mkdir(parents=True, exist_ok=True)
            
            for p in cls_dir.glob('*.avi'):
                out_path = out_split / cls / f"{p.stem}.npy"
                tasks.append((str(p), str(out_path)))

    def worker(paths):
        video_path, save_path = paths
        frames = extract_frames(video_path, NUM_FRAMES, FRAME_H, FRAME_W)
        if frames is not None:
            np.save(save_path, frames) 

    # Use ThreadPool to saturate CPU (adjust max_workers based on Kaggle cores)
    with ThreadPoolExecutor(max_workers=4) as executor:
        list(tqdm(executor.map(worker, tasks), total=len(tasks), desc="Processing"))

    print("Preprocessing complete!")

# Run it
preprocess_dataset()

Found existing preprocessed data at ./preprocessed_16_224_224. Skipping.


## Dataset Class & Train/Val/Test Split

In [ ]:
import random
import torch
from torch.utils.data import Dataset, DataLoader

import torch
from torch.utils.data import Dataset
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm


class RAMDataset(Dataset):
    """
    Optimized for .npy files and Kaggle's 30GB RAM.
    Loads data as float16 to save space, casts to float32 on the fly.
    """
    def __init__(self, split, augment=False):
        self.split = split
        self.augment = augment
        self.samples = []
        
        # 1. Collect all .npy paths
        base_path = Path(PREPROCESSED_DIR) / split
        
        raw_samples = []
        for cls in CLASS_NAMES:
            cls_dir = base_path / cls
            if not cls_dir.exists(): continue
            for p in cls_dir.glob('*.npy'):
                raw_samples.append((p, CLASS_TO_IDX[cls]))

        print(f"Loading {split} split into RAM (Total: {len(raw_samples)} clips)...")
        
        # 2. Bulk load and pre-process
        for path, label in tqdm(raw_samples, desc=f"RAM {split}", unit="clip"):
            
            # Load .npy (Shape: NUM_FRAMES, H, W, 3 | dtype: uint8)
            frames = np.load(path)            
            # Normalization (Matches your previous logic)
            frames = frames.astype(np.float32) / 255.0
            frames = (frames - MEAN) / STD
            frames = frames.transpose(0, 3, 1, 2) # (T, C, H, W)
            
            # Convert to float16 to fit in Kaggle RAM (30GB limit)
            clip_tensor = torch.from_numpy(frames).to(torch.float16)
            self.samples.append((clip_tensor, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        clip, label = self.samples[i]
        
        # Cast back to float32 for the model
        clip = clip.to(torch.float32)

        # # Augmentation: Horizontal Flip (Simple & Effective for Cricket)
        if self.augment and random.random() < 0.5:
            clip = torch.flip(clip, dims=[-1])
            
        return clip, label


class ActionDataset(Dataset):
    """
    Loads individual .npy files from disk.
    Memory efficient: only BATCH_SIZE * NUM_WORKERS clips are in RAM at once.
    """
    def __init__(self, split, augment=False):
        self.split = split
        self.augment = augment
        self.samples = []
        
        # Collect paths and labels 
        base = Path(PREPROCESSED_DIR) / split
        for cls in CLASS_NAMES:
            cls_dir = base / cls
            if not cls_dir.exists(): continue
            for p in cls_dir.glob('*.npy'):
                self.samples.append((str(p), CLASS_TO_IDX[cls]))
        
        print(f"Dataset {split}: Found {len(self.samples)} clips.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        
        frames = np.load(path) 

        frames = frames.astype(np.float32) / 255.0
        frames = (frames - MEAN) / STD
        frames = frames.transpose(0, 3, 1, 2)
        
        clip = torch.from_numpy(frames)
        
        if self.augment and random.random() < 0.5:
            clip = torch.flip(clip, dims=[-1])

        return clip, label

def make_loader(split, augment=False, shuffle=False):
    dataset = RAMDataset(split, augment=augment)
    
    return DataLoader(
        dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = shuffle,
        num_workers=0, 
        pin_memory=True 
    )

train_loader = make_loader('train', augment=True,  shuffle=True)
val_loader   = make_loader('val',   augment=False, shuffle=True)
test_loader  = make_loader('test',  augment=False, shuffle=True)

print(f"Batches → train:{len(train_loader)}  val:{len(val_loader)}  test:{len(test_loader)}")


## Model Architecture

In [10]:
import torch.nn as nn
import torch.nn.functional as F

import torch.nn as nn
import torch.nn.functional as F

import torch.nn as nn


class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        ch = CNN_CHANNELS

        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, ch[0], 3, padding=1),
            nn.BatchNorm2d(ch[0]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 112 -> 56
            nn.Conv2d(ch[0], ch[1], 3, padding=1),
            nn.BatchNorm2d(ch[1]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Dropout2d(p=0.2),

            # Block 3: 56 -> 28
            nn.Conv2d(ch[1], ch[2], 3, padding=1),
            nn.BatchNorm2d(ch[2]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4: 28 -> 14
            nn.Conv2d(ch[2], ch[3], 3, padding=1),
            nn.BatchNorm2d(ch[3]), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Dropout2d(p=0.2)
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.project = nn.Linear(ch[3] * 4*4, CNN_PROJ_DIM)

    def forward(self, x):
        x = self.features(x)

        x = self.pool(x).flatten(1)
        return self.project(x)

class LSTMEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=CNN_PROJ_DIM,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(p=0.3)
        self._init_weights()
        self.attn_score = nn.Linear(LSTM_HIDDEN * 2, 1)

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out) 
        scores = self.attn_score(out)   # (B, T, 1)
        weights = torch.softmax(scores, dim=1)  # (B, T, 1)
        context = (out * weights).sum(dim=1)    # (B, H*2)        
        return context
        
class ClassifierHead(nn.Module):
    """
    Input  : (B, 256)
    Output : (B, num_classes)  logits
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LSTM_HIDDEN*2, CLF_HIDDEN),
            nn.ReLU(),
            nn.Dropout(CLF_DROPOUT),
            nn.Linear(CLF_HIDDEN, NUM_CLASSES),
        )

    def forward(self, x):
        return self.net(x)
        

class CricketActionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn  = CNNEncoder()
        self.lstm = LSTMEncoder()
        self.head = ClassifierHead()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        feat = self.cnn(x.view(B * T, C, H, W)).view(B, T, -1)
        context = self.lstm(feat)
        return self.head(context)



# ── Sanity check ──────────────────────────────────────────────────────────────
model  = CricketActionNet().to(DEVICE)
# Wrap the model for multi-GPU use
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
_x     = torch.zeros(2, NUM_FRAMES, 3, FRAME_H, FRAME_W).to(DEVICE)
_out   = model(_x)
print(f"Output shape : {_out.shape} ")
print(f"Parameters   : {sum(p.numel() for p in model.parameters()):,}")
del _x, _out

try:
    from torchinfo import summary
    summary(model, input_size=(2, NUM_FRAMES, 3, FRAME_H, FRAME_W),
            col_names=['input_size','output_size','num_params'], depth=3, verbose=1)
except Exception as e:
    print(e)


Output shape : torch.Size([2, 10]) 
Parameters   : 2,558,347
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
CricketActionNet                         [2, 16, 3, 224, 224]      [2, 10]                   --
├─CNNEncoder: 1-1                        [32, 3, 224, 224]         [32, 256]                 --
│    └─Sequential: 2-1                   [32, 3, 224, 224]         [32, 256, 14, 14]         --
│    │    └─Conv2d: 3-1                  [32, 3, 224, 224]         [32, 32, 224, 224]        896
│    │    └─BatchNorm2d: 3-2             [32, 32, 224, 224]        [32, 32, 224, 224]        64
│    │    └─ReLU: 3-3                    [32, 32, 224, 224]        [32, 32, 224, 224]        --
│    │    └─MaxPool2d: 3-4               [32, 32, 224, 224]        [32, 32, 112, 112]        --
│    │    └─Conv2d: 3-5                  [32, 32, 112, 112]        [32, 64, 112, 112]        18,496
│    │    └─BatchNorm2d: 3-6             [32, 64, 112, 112]      

## Minimal Logger

In [50]:
import json, shutil, time as _time

EPOCH_LOG = f'{LOG_DIR}/epoch_log.json'
BATCH_LOG = f'{LOG_DIR}/batch_log.json'

def _write(path, data):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(data, f)
    shutil.move(tmp, path)

def init_logs():
    _write(EPOCH_LOG, {'epochs': []})
    _write(BATCH_LOG, {'batches': []})
    print("Logs initialised →", LOG_DIR)

def log_epoch(epoch, train_loss, val_loss, train_acc, val_acc, lr, duration_s):
    with open(EPOCH_LOG) as f: data = json.load(f)
    data['epochs'].append({
        'epoch':      epoch,
        'train_loss': round(float(train_loss), 6),
        'val_loss':   round(float(val_loss),   6),
        'train_acc':  round(float(train_acc),  4),
        'val_acc':    round(float(val_acc),    4),
        'lr':         float(lr),
        'duration_s': round(duration_s, 1),
    })
    _write(EPOCH_LOG, data)

def log_batch(epoch, batch_idx, loss, acc):
    try:
        with open(BATCH_LOG) as f: data = json.load(f)
    except Exception:
        data = {'batches': []}
    data['batches'].append({
        'e': epoch, 'b': batch_idx,
        'loss': round(float(loss), 5), 'acc': round(float(acc), 4),
    })
    data['batches'] = data['batches'][-500:]   # keep last 500 batches
    _write(BATCH_LOG, data)

init_logs()


Logs initialised → /kaggle/working/logs


## Training Loop

In [51]:
from tqdm.notebook import tqdm, trange
import time as _time
import torchvision.transforms.v2 as T


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, amsgrad=False, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_loss  = float('inf')

gpu_augmentor = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
])

scaler = torch.amp.GradScaler('cuda')

def run_phase(loader, train: bool):
    """One full pass over train or val set. Returns (loss, accuracy)."""
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()

    # Inner tqdm for batches — leave=False so it disappears after each epoch
    batch_bar = tqdm(loader, leave=False, unit='batch', dynamic_ncols=True,
                     desc='  train' if train else '  val  ')

    with ctx:
        for batch_idx, (clips, labels) in enumerate(batch_bar):
            clips  = clips.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)
                # with torch.no_grad():
                #     clips = gpu_augmentor(clips)

            with torch.amp.autocast('cuda'):
                logits = model(clips)
                loss   = criterion(logits, labels)

            if train:
                
                # --- Mixed Precision Backward Pass ---
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()


            bs          = labels.size(0)
            loss_val    = loss.item()
            total_loss += loss_val * bs
            preds       = logits.argmax(1)
            correct    += (preds == labels).sum().item()
            total      += bs

            batch_acc = (preds == labels).float().mean().item()

            # Live postfix on the batch bar
            batch_bar.set_postfix(
                loss=f'{loss_val:.4f}',
                acc=f'{batch_acc:.3f}',
                refresh=False,
            )

            if train:
                log_batch(current_epoch, batch_idx, loss_val, batch_acc)

    return total_loss / total, correct / total


# ─── Outer epoch loop ─────────────────────────────────────────────────────────
epoch_bar = trange(1, NUM_EPOCHS + 1, desc='Epochs', unit='epoch', dynamic_ncols=True)

for epoch in epoch_bar:
    current_epoch = epoch
    t0 = _time.perf_counter()
    
    train_loss, train_acc = run_phase(train_loader, train=True)
    # val_loss, val_acc = run_phase(sample_loader, train=True)
    val_loss,   val_acc   = run_phase(val_loader,   train=False)

    scheduler.step()
    lr       = optimizer.param_groups[0]['lr']
    duration = _time.perf_counter() - t0

    log_epoch(epoch, train_loss, val_loss, train_acc, val_acc, lr, duration)

    # ── Outer bar postfix ──────────────────────────────────────────────────────
    epoch_bar.set_postfix(
        tl=f'{train_loss:.4f}',
        vl=f'{val_loss:.4f}',
        ta=f'{train_acc:.3f}',
        va=f'{val_acc:.3f}',
        lr=f'{lr:.1e}',
        refresh=False,
    )

    # ── Best model ────────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss':        val_loss,
            'val_acc':         val_acc,
            'class_names':     CLASS_NAMES,
        }, f'{CKPT_DIR}/best_model.pth')

    # ── Periodic checkpoint ────────────────────────────────────────────────────
    if epoch % CHECKPOINT_EVERY == 0:
        torch.save({
            'epoch':            epoch,
            'model_state':      model.state_dict(),
            'optimizer_state':  optimizer.state_dict(),
            # 'scheduler_state':  scheduler.state_dict(),
            'val_loss':         val_loss,
        }, f'{CKPT_DIR}/epoch_{epoch:03d}.pth')

print(f"\nTraining complete.  Best val loss: {best_val_loss:.4f}")


Epochs:   0%|          | 0/100 [00:00<?, ?epoch/s]

  train:   0%|          | 0/79 [00:00<?, ?batch/s]

  val  :   0%|          | 0/16 [00:00<?, ?batch/s]

  train:   0%|          | 0/79 [00:00<?, ?batch/s]

  val  :   0%|          | 0/16 [00:00<?, ?batch/s]

...

  val  :   0%|          | 0/16 [00:00<?, ?batch/s]

  train:   0%|          | 0/79 [00:00<?, ?batch/s]

  val  :   0%|          | 0/16 [00:00<?, ?batch/s]

  train:   0%|          | 0/79 [00:00<?, ?batch/s]

KeyboardInterrupt: 